In [1]:
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [2]:
data = load_breast_cancer()

X = data.data
y = data.target

print(X.shape)
print(y.shape)

(569, 30)
(569,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [5]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [6]:
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
).reshape(-1, 1)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)

In [7]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [8]:
class ANN(nn.Module):

    def __init__(self, input_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

model = ANN(X_train.shape[1])

print(model)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    weight_decay=0.01
)

ANN(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [9]:
epochs = 100

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        # Forward pass
        outputs = model(X_batch)

        # Data loss
        loss = criterion(outputs, y_batch)

        # Clear gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights + L2 regularization
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {avg_loss:.4f}"
        )

Epoch [10/100] Loss: 0.5878
Epoch [20/100] Loss: 0.4292
Epoch [30/100] Loss: 0.2589
Epoch [40/100] Loss: 0.1799
Epoch [50/100] Loss: 0.1425
Epoch [60/100] Loss: 0.1163
Epoch [70/100] Loss: 0.0978
Epoch [80/100] Loss: 0.0933
Epoch [90/100] Loss: 0.0820
Epoch [100/100] Loss: 0.0742


In [10]:
model.eval()

with torch.no_grad():

    val_logits = model(X_val_tensor)

    val_probabilities = torch.sigmoid(val_logits)

    val_predictions = (
        val_probabilities >= 0.5
    ).float()

In [11]:
val_accuracy = accuracy_score(
    y_val_tensor.numpy(),
    val_predictions.numpy()
)

print(f"Validation Accuracy: {val_accuracy:.4f}")

Validation Accuracy: 0.9780


In [12]:
model.eval()

with torch.no_grad():

    test_logits = model(X_test_tensor)

    test_probabilities = torch.sigmoid(test_logits)

    test_predictions = (
        test_probabilities >= 0.5
    ).float()

test_accuracy = accuracy_score(
    y_test_tensor.numpy(),
    test_predictions.numpy()
)

print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 0.9649
